# 03 · The Hero cascade — d8 global + EBM regime = the 0.12033 ceiling

*Edge-features arc · 01 discovery · 02 linear base · 03 hero cascade · 04 regime-MoE · 05 temporal-kNN  —  machinery: `ridge_pipeline_throughline.ipynb`*

**Verify first.** The deployed stack is a 3-stage cascade: `linbest` base + **d8 global XGB** (Hero A) +
**gated EBM regime** on the h16-19 close/AH leftover (Hero B). Every tree-stage QLIKE below is
**recomputed from the saved per-bar predictions through the real pipeline metric**
(`apply_duan_smearing` → QLIKE) and **asserted** against the cluster collect (`results_all.csv`) — never
pasted. The base row is the linear floor (no local preds), so it carries its exact **reproduce-command**
instead. Machinery = `resid_amortized.preds_chunk` (`resid_regime` arm, the through-line for ch. 04).

In [1]:
import html, inspect, json, os, sys, textwrap
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

def find_repo(s):
    for q in [Path(s).resolve(), *Path(s).resolve().parents]:
        if (q / "resid_amortized.py").exists() and (q / "src").is_dir():
            return q
    raise FileNotFoundError("repo root")
REPO = find_repo(Path.cwd()); os.chdir(REPO); sys.path.insert(0, str(REPO))

from src.evaluation.metrics import apply_duan_smearing  # the REAL pipeline metric

# Collapsible, theme-following source display (one <details> per object; folded).
def _details(f, open_=False):
    mod = f.__module__.replace("src.", "src/").replace(".", "/") + ".py"
    try:
        sig = ("class " + f.__name__) if inspect.isclass(f) else ("def " + f.__name__ + str(inspect.signature(f)))
    except (ValueError, TypeError):
        sig = f.__qualname__
    body = "```python\n" + textwrap.dedent(inspect.getsource(f)).rstrip() + "\n```"
    return (f"<details{' open' if open_ else ''}>\n<summary><code>{html.escape(mod + '  ·  ' + sig)}"
            f"</code></summary>\n\n{body}\n\n</details>")
def show_one(f):
    return Markdown(_details(f))

# ── the verification machinery: recompute QLIKE from saved per-bar preds ──────
# Local preds are gitignored; if absent we print the exact reproduce-command.
PREDS = REPO / "results" / "moe_ladder" / "preds"
CID = "xgb_all_buckets_tw1000_enetreg2_linbest_rf480_slim"

def recompute(label):
    """Full-OOS QLIKE recomputed from saved per-bar preds via apply_duan_smearing.
    Returns (qlike, n) if the preds csv is present, else (None, 0)."""
    fp = PREDS / f"{label}.csv"
    if not fp.exists():
        return None, 0
    df = pd.read_csv(fp)                                  # cols: k, pred_adj, y_true, base
    pr, tr = apply_duan_smearing(df.pred_adj.to_numpy(), df.y_true.to_numpy(), df.base.to_numpy())
    m = (tr > 0) & (pr > 0); r = tr[m] / pr[m]
    return float(np.mean(r - np.log(r) - 1.0)), int(len(df))

def reproduce_cmd(arm, label):
    return f"$PY resid_amortized.py chunk_collect {CID} {arm} {label}"

# cluster reference (results_all.csv) — what each RECOMPUTE must reproduce to <1e-4
REF = pd.read_csv(REPO / "results" / "moe_ladder" / "results_all.csv").set_index("label")
print("setup ok | preds present:", PREDS.exists(), "| reference rows:", len(REF))

setup ok | preds present: True | reference rows: 11


---
## 1 · Verify — the cascade ladder, recomputed

The thin through-line: each tree-stage row reloads its saved per-bar predictions and runs the **exact**
pipeline QLIKE on them — `apply_duan_smearing(pred_adj, y_true, base)` → masked `ratio − log ratio − 1`
mean (folded below). The recompute is asserted against the cluster `results_all.csv` to `<1e-4`. The base
(`enetreg2_linbest`, the linear floor) is not a residual collect and has no local preds, so its row shows
the **reproduce-command** and reads its value from the recorded `linbest_ladder.csv` artifact (not pasted
into this cell).

In [2]:
show_one(apply_duan_smearing)   # the real metric — folded, recomputed below

<details>
<summary><code>src/evaluation/metrics.py  ·  def apply_duan_smearing(forecasts: &#x27;np.ndarray&#x27;, y_true: &#x27;np.ndarray&#x27;, baselines: &#x27;np.ndarray&#x27;) -&gt; &#x27;tuple[np.ndarray, np.ndarray]&#x27;</code></summary>

```python
def apply_duan_smearing(
    forecasts: np.ndarray,
    y_true: np.ndarray,
    baselines: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Apply Duan smearing correction to convert adjusted-scale forecasts to raw scale.

    Parameters
    ----------
    forecasts : array-like
        Model predictions on adjusted (sqrt / log) scale.
    y_true : array-like
        True values on adjusted scale.
    baselines : array-like
        Baseline volatility used to scale back to raw units.

    Returns
    -------
    pred_raw : np.ndarray
        Smearing-corrected predictions on raw scale.
    true_raw : np.ndarray
        True values on raw scale.
    """
    forecasts = np.asarray(forecasts, dtype=np.float64)
    y_true = np.asarray(y_true, dtype=np.float64)
    baselines = np.asarray(baselines, dtype=np.float64)

    smear = np.mean((y_true - forecasts) ** 2)
    pred_raw = (forecasts**2 + smear) * baselines
    true_raw = (y_true**2) * baselines
    return pred_raw, true_raw
```

</details>

In [3]:
# base value comes from the recorded ladder artifact, not hardcoded here
LL = pd.read_csv(REPO / "results" / "moe_ladder" / "linbest_ladder.csv")
base_q = float(LL.loc[LL.label == "enetreg2_linbest", "qlike"].iloc[0])

# (stage, preds-label | None, arm, note).  None preds-label ⇒ cluster-only ⇒ reproduce-cmd.
LADDER = [
    ("base  enetreg2_linbest",          None,       "residualized", "linear floor — no local preds"),
    ("+ d8@cs0.5 global XGB (Hero A)",   "fa_d8c5",  "heroA",        "global tree, no regime stage"),
    ("+ tuned XGB regime (Hero B′)",     "fb_r3n4",  "resid_regime", "best curated XGB regime (depth 3)"),
    ("+ EBM regime  =  THE CEILING",     "ctrl_ebm", "resid_regime", "EBM regime — the bar (== lbHeroB)"),
]
rows = []
for stage, lbl, arm, note in LADDER:
    if lbl is None:
        rows.append({"stage": stage, "qlike": round(base_q, 5), "n": -1,
                     "source": "cluster (reproduce-cmd)", "ref/cmd": reproduce_cmd(arm, "<linbest base>"),
                     "note": note})
        continue
    q, n = recompute(lbl)
    assert q is not None, f"{lbl}: preds missing — reproduce with  {reproduce_cmd(arm, lbl)}"
    ref = float(REF.loc[lbl, "qlike_full"])
    assert abs(q - ref) < 1e-4, f"{lbl}: RECOMPUTE {q:.5f} != cluster {ref:.5f}"
    rows.append({"stage": stage, "qlike": round(q, 5), "n": n,
                 "source": "RECOMPUTE", "ref/cmd": f"== results_all {ref:.5f}", "note": note})

# anchor identity: ctrl_ebm must reproduce the INDEPENDENT lbHeroB collect
q_ctrl, _ = recompute("ctrl_ebm"); q_lb, _ = recompute("lbHeroB")
assert q_ctrl is not None and q_lb is not None, "anchor preds missing"
assert abs(q_ctrl - q_lb) < 1e-4, f"ctrl_ebm {q_ctrl:.5f} != lbHeroB {q_lb:.5f}"

ladder = pd.DataFrame(rows)
display(ladder[["stage", "qlike", "source", "ref/cmd", "n", "note"]])
print(f"PASS — every RECOMPUTE row reproduces its cluster QLIKE to <1e-4; "
      f"ctrl_ebm reproduces the independent lbHeroB collect ({q_ctrl:.5f} vs {q_lb:.5f}).")
print(f"Ceiling = {q_ctrl:.5f}  (linbest + d8@cs0.5 global + EBM regime). ch.04 targets exactly this slot.")
print(f"Base (linear floor, no preds) — reproduce: {reproduce_cmd('residualized', '<linbest base>')}")

,stage,qlike,source,ref/cmd,n,note
0,base enetreg2_linbest,0.12266,cluster (reproduce-cmd),$PY resid_amortized.py chunk_collect xgb_all_b...,-1,linear floor — no local preds
1,+ d8@cs0.5 global XGB (Hero A),0.12081,RECOMPUTE,== results_all 0.12081,194934,"global tree, no regime stage"
2,+ tuned XGB regime (Hero B′),0.12072,RECOMPUTE,== results_all 0.12072,194934,best curated XGB regime (depth 3)
3,+ EBM regime = THE CEILING,0.12033,RECOMPUTE,== results_all 0.12033,194934,EBM regime — the bar (== lbHeroB)


PASS — every RECOMPUTE row reproduces its cluster QLIKE to <1e-4; ctrl_ebm reproduces the independent lbHeroB collect (0.12033 vs 0.12033).
Ceiling = 0.12033  (linbest + d8@cs0.5 global + EBM regime). ch.04 targets exactly this slot.
Base (linear floor, no preds) — reproduce: $PY resid_amortized.py chunk_collect xgb_all_buckets_tw1000_enetreg2_linbest_rf480_slim residualized <linbest base>


**Source — the cluster pipeline that produced the per-bar predictions.** The tree-stage QLIKEs above are recomputed *locally* from saved per-bar preds through `apply_duan_smearing` (folded just above), but those preds themselves were produced on the cluster by `resid_amortized.preds_chunk` (the `resid_regime` arm) on the linbest cache — value from the CARC linbest run (cluster-only cache); source shown here. The 3-stage cascade — linbest base + global d8 XGB (Hero A) on the resid_subset survivors + gated EBM regime on the h16-19 close/AH leftover (Hero B) — and the tree factory it builds each stage with are folded from source:

<details>
<summary><code>resid_amortized.py :: preds_chunk</code></summary>

```python
def preds_chunk(cache, arm, cfg, blk0, blk1):
    """Residualized OOS preds for the rows covered by cadence blocks [blk0, blk1).

    Chunks partition the cadence blocks, so each tree-refit point is fit in exactly ONE
    chunk (no repeated work) and the trial's full preds are the ordered concat of chunks.
    Returns (k0, k1, preds) with k = t - train_win (OOS index)."""
    c = cache
    tw = c["cell"]["train_win"]
    model = c["cell"]["model"]
    n = len(c["Xs"])
    starts = c["starts"]
    k0 = int(starts[blk0]) - tw
    k1 = (int(starts[blk1]) if blk1 < len(starts) else n) - tw
    if arm == "ridge_alone":
        return k0, k1, np.array(c["ridge_oos"][k0:k1], copy=True)
    if arm == "resid_regime":
        # Hero B regime cascade, per cadence block i:
        #   out = base_oos + GLOBAL_tree(survivors) + gated REGIME_tree(leftover, h16-19 only)
        # GLOBAL stage = Hero-A winning XGB (env GLOBAL_CFG) on the resid_subset survivors masks[i],
        # fit on r1 = y_train - base_train (the residualized target). REGIME stage = EBM (env
        # REGIME_CFG) on the LEFTOVER r2 = r1 - global(Xtr), trained on h16-19 TRAIN rows only and
        # predicting ZERO outside h16-19. With REGIME_CFG empty ({} / unset) the regime stage is
        # skipped, so this arm reproduces the single-pass resid_subset number (SANITY invariant --
        # given the resid_subset run uses the SAME xgb cfg as GLOBAL_CFG on an xgb cell).
        if "masks" not in c:
            raise KeyError(
                "resid_regime needs per-block enet survivor masks (run enet_masks first)"
            )
        if c.get("feats") is None:
            raise KeyError(
                "resid_regime needs aligned feats (cell feats.json) for the hour gate"
            )
        out = np.array(c["ridge_oos"][k0:k1], copy=True)
        masks = c["masks"]
        fm = c.get(
            "force_mask"
        )  # FORCE_COLS: union into survivors so the tree/EBM see them
        hr = c["Xs"][:, c["feats"].index("hour")]
        re = c.get(
            "regime_extra"
        )  # REGIME_EXTRA: regime-persistence features injected into the EBM ONLY -- they BYPASS the
        # global enet base (unchanged ridge_oos/coefs) and the global d8 tree (g fits survivors only),
        # entering at the regime stage where there is no global alpha to mis-penalize them. None -> off.
        gcfg = json.loads(os.environ.get("GLOBAL_CFG", "{}"))
        rcfg = json.loads(os.environ.get("REGIME_CFG", "{}"))
        # regime-stage learner: default EBM (interpretable); REGIME_MODEL=xgb -> tuned XGB for PURE
        # POWER (drops the EBM additive/pairwise constraint = the QLIKE ceiling, no interpretability).
        rmodel = os.environ.get("REGIME_MODEL", "ebm")
        regime_full = os.environ.get("REGIME_FULL", "0") not in ("", "0")  # FREE the gate (#1): fit/predict
        # on ALL hours so the learned MoE gate discovers the regime, vs the hand h16-19 pre-gate (default).
        # REGIME_INVERT (#3, structural-starvation test): swap the cascade order to REGIME-then-global, so
        # the soft-routing MoE gets FIRST crack at the un-starved r1 = y - base and d8 only soaks up its
        # leftover. Default (off) = global-then-regime (d8 eats the X-routable structure -> gate starved).
        invert = os.environ.get("REGIME_INVERT", "0") not in ("", "0")
        # REGIME_NOGLOBAL: drop the d8 global stage entirely -> base + regime ONLY (the purest un-starved
        # cascade: additive base + gate-as-sole-regime, no hard router competing). Only honored with invert.
        skip_global = os.environ.get("REGIME_NOGLOBAL", "0") not in ("", "0")
        hr_idx = c["feats"].index("hour")
        mk_g = _tree_factory("xgb", gcfg)
        for i in range(blk0, blk1):
            t_r = int(starts[i])
            cols = masks[i] if fm is None else (masks[i] | fm)
            Xtr = c["Xs"][t_r - tw : t_r]
            t_end = int(starts[i + 1]) if i + 1 < len(starts) else n
            r1 = c["y"][t_r - tw : t_r] - (Xtr @ c["coefs"][i] + c["intercepts"][i])
            if invert:  # REGIME FIRST on the un-starved r1 (gate's first crack), then d8 mops up the leftover
                Xblk = c["Xs"][t_r:t_end][:, cols]
                m_tr = _close_mask(hr[t_r - tw : t_r])
                pe_tr = np.zeros_like(r1)
                if rcfg and (regime_full or int(m_tr.sum()) >= 50):
                    cols_e = cols
                    if regime_full:
                        cols_e = cols.copy()
                        cols_e[hr_idx] = True
                    Xtr_e = Xtr[:, cols_e]
                    Xblk_e = c["Xs"][t_r:t_end][:, cols_e]
                    if re is not None:
                        Xtr_e = np.hstack([Xtr_e, re[t_r - tw : t_r]])
                        Xblk_e = np.hstack([Xblk_e, re[t_r:t_end]])
                    e = _tree_factory(rmodel, rcfg)()
                    if regime_full:
                        e.fit(Xtr_e, r1)
                        out[t_r - tw - k0 : t_end - tw - k0] += e.predict(Xblk_e).ravel()
                        pe_tr = e.predict(Xtr_e).ravel()
                    else:
                        e.fit(Xtr_e[m_tr], r1[m_tr])
                        pe = e.predict(Xblk_e).ravel()
                        pe[~_close_mask(hr[t_r:t_end])] = 0.0
                        out[t_r - tw - k0 : t_end - tw - k0] += pe
                        pe_tr = e.predict(Xtr_e).ravel()
                        pe_tr[~m_tr] = 0.0
                if not skip_global:  # d8 soaks up the regime leftover (off -> base + regime only)
                    g = mk_g()
                    g.fit(Xtr[:, cols], r1 - pe_tr)
                    out[t_r - tw - k0 : t_end - tw - k0] += g.predict(Xblk).ravel()
                continue
            g = mk_g()
            g.fit(Xtr[:, cols], r1)
            Xblk = c["Xs"][t_r:t_end][:, cols]
            out[t_r - tw - k0 : t_end - tw - k0] += g.predict(Xblk).ravel()
            if (
                not rcfg
            ):  # regime stage disabled -> single-pass resid_subset (sanity invariant)
                continue
            m_tr = _close_mask(hr[t_r - tw : t_r])
            if (
                not regime_full and int(m_tr.sum()) < 50
            ):  # too few close/AH train rows -> skip regime this block (predict 0)
                continue
            r2 = r1 - g.predict(Xtr[:, cols]).ravel()
            cols_e = cols
            if regime_full:  # add `hour` so the freed gate can route on the clock itself
                cols_e = cols.copy()
                cols_e[hr_idx] = True
            Xtr_e = Xtr[:, cols_e]
            Xblk_e = c["Xs"][t_r:t_end][:, cols_e]
            if re is not None:  # regime model also sees the persistence extras (global g unaffected above)
                Xtr_e = np.hstack([Xtr_e, re[t_r - tw : t_r]])
                Xblk_e = np.hstack([Xblk_e, re[t_r:t_end]])
            if rmodel == "ebm_mtfm":
                # EBM (+) multi-task-FM ENSEMBLE: the binned-bagged EBM on r2 plus an un-starved multi-task
                # FM (aux head predicts r1, the pre-d8 residual). They are DIVERSE (pre-check corr ~0.34) so
                # the weighted average can beat the EBM alone. ENS_W = weight on the EBM; MT_* = the FM knobs.
                from src.models.regime_moe import MultiTaskFM

                ens_w = float(os.environ.get("ENS_W", "0.4"))
                # un-starve the ANTICIPATION: MT_GATE=ghat -> per-row aux weight gated by |ghat|=|r1-r2|
                # (d8's bite, the taken structure made explicit) — spend the aux where d8 took a big bite.
                gate_kind = os.environ.get("MT_GATE", "uniform")
                aux_hi = float(os.environ.get("MT_AUXHI", "0.6"))
                ghat = r1 - r2  # = g.predict(Xtr[:, cols]); causal, available at train

                def _auxw(sel):
                    if gate_kind != "ghat":
                        return None
                    b = np.abs(ghat[sel])
                    return np.where(b > np.median(b), aux_hi, 0.0).astype(np.float32)

                ebm = _tree_factory("ebm", json.loads(os.environ.get("EBM_CFG", "{}")))()
                mt = MultiTaskFM(
                    rank=int(os.environ.get("MT_RANK", "4")),
                    n_bags=int(os.environ.get("MT_NBAGS", "8")),
                    epochs=int(os.environ.get("MT_EPOCHS", "250")),
                    aux_weight=float(os.environ.get("MT_AUXW", "0.3")),
                    weight_decay=float(os.environ.get("MT_WD", "0.1")),
                )
                if regime_full:
                    ebm.fit(Xtr_e, r2)
                    mt.fit(Xtr_e, r2, r1, aux_w=_auxw(slice(None)))
                    pe = (ens_w * ebm.predict(Xblk_e) + (1 - ens_w) * mt.predict(Xblk_e)).ravel()
                else:
                    ebm.fit(Xtr_e[m_tr], r2[m_tr])
                    mt.fit(Xtr_e[m_tr], r2[m_tr], r1[m_tr], aux_w=_auxw(m_tr))
                    pe = (ens_w * ebm.predict(Xblk_e) + (1 - ens_w) * mt.predict(Xblk_e)).ravel()
                    pe[~_close_mask(hr[t_r:t_end])] = 0.0
            else:
                e = _tree_factory(rmodel, rcfg)()
                if regime_full:  # fit/predict on ALL hours; the learned gate discovers WHERE the regime is
                    e.fit(Xtr_e, r2)
                    pe = e.predict(Xblk_e).ravel()
                else:  # hand pre-gate: fit on h16-19 train rows, predict gated to h16-19
                    e.fit(Xtr_e[m_tr], r2[m_tr])
                    pe = e.predict(Xblk_e).ravel()
                    pe[~_close_mask(hr[t_r:t_end])] = 0.0
            out[t_r - tw - k0 : t_end - tw - k0] += pe
        return k0, k1, out
    raw = arm == "raw_tree"
    out = np.zeros(k1 - k0) if raw else np.array(c["ridge_oos"][k0:k1], copy=True)
    if (
        arm == "resid_subset"
    ):  # tree sees only the block's rolling enet survivors (~120)
        fm = c.get("force_mask")  # FORCE_COLS: union forced cols into the survivor set

        def colsel(i):
            return c["masks"][i] if fm is None else (c["masks"][i] | fm)
    elif (
        arm == "resid_subset_ind"
    ):  # survivors UNION live availability indicators (event channel)
        li = c["live_ind"]

        def colsel(i):
            return c["masks"][i] if li is None else (c["masks"][i] | li)
    elif (
        arm == "resid_subset_nocov"
    ):  # survivors MINUS coverage-artifact indicators (voldemand-type
        cov = c.get(
            "cov_mask"
        )  # availability steps) -> tests if the EBM leans on a data artifact

        def colsel(i):
            return c["masks"][i] if cov is None else (c["masks"][i] & ~cov)
    elif (
        arm == "resid_pruned"
    ):  # tree sees all-but-the-signalless (the 224-indicator prune)
        keep = ~c["prunable"]

        def colsel(i):
            return keep
    else:  # residualized / raw_tree: tree sees all features

        def colsel(i):
            return slice(None)

    mk = _tree_factory(model, cfg)
    for i in range(blk0, blk1):
        t_r = int(starts[i])
        Xtr = c["Xs"][t_r - tw : t_r]
        cols = colsel(i)
        r_train = (
            c["y"][t_r - tw : t_r]
            if raw
            else c["y"][t_r - tw : t_r] - (Xtr @ c["coefs"][i] + c["intercepts"][i])
        )
        tree = mk()
        tree.fit(Xtr[:, cols], r_train)
        t_end = int(starts[i + 1]) if i + 1 < len(starts) else n
        out[t_r - tw - k0 : t_end - tw - k0] += tree.predict(
            c["Xs"][t_r:t_end][:, cols]
        ).ravel()
    return k0, k1, out
```

</details>

<details>
<summary><code>resid_amortized.py :: _tree_factory</code></summary>

```python
def _tree_factory(model, cfg):
    if model in ("lgbm", "lightgbm"):
        from lightgbm import LGBMRegressor

        kw = dict(cfg)
        kw.setdefault("n_jobs", 4)
        kw.setdefault("verbose", -1)
        kw.setdefault("random_state", 42)
        return lambda: LGBMRegressor(**kw)
    if (
        model == "ebm"
    ):  # Microsoft EBM: bagged boosted GAM (additive + pairwise), interpretable
        from interpret.glassbox import ExplainableBoostingRegressor

        kw = dict(cfg)
        kw.setdefault("n_jobs", 4)
        kw.setdefault("random_state", 42)
        return lambda: ExplainableBoostingRegressor(**kw)
    if (
        model == "moe"
    ):  # interpretable regime mixture-of-experts (learned soft-tree gate + NAM experts)
        from src.models.regime_moe import RegimeMoE

        return lambda: RegimeMoE(**dict(cfg))
    from xgboost import XGBRegressor

    kw = dict(cfg)
    kw.setdefault("n_jobs", 4)
    kw.setdefault("tree_method", "hist")
    kw.setdefault("random_state", 42)
    return lambda: XGBRegressor(**kw)
```

</details>

---
## 2 · Interpret — the EBM regime is the *correct* learner, not a compromise

The ladder is an **expressivity stack that subsumes upward**: each stage only earns its keep where the
stage below it left structure on the table.

- **The global d8 tree subsumes the linear base.** linbest `0.12266` → +d8@cs0.5 global **`0.12081`**
  (−0.00185). One global XGB absorbs the bulk of the residual nonlinearity the penalized-linear base
  could not — the famous edge is a *global* tree effect, not a regime one.
- **The XGB regime barely moves, and LOSES to the EBM.** A tuned XGB *regime* stage on the h16-19
  close/AH leftover gives only **`0.12072`** (−0.00009 vs no regime), and the proper **120-trial Optuna**
  search lands at 0.12094 — *worse than doing nothing* (the no-regime global, 0.12081). On the
  few-thousand-row, ~93%-noise h16-19 subset a powerful booster overfits; confirmed by optimization,
  not assertion.
- **The EBM regime is the right regularizer.** Swapping XGB→EBM on the *same* slot gives **`0.12033`**
  (−0.00048 vs no-regime), reproducing the independent `lbHeroB` collect. The EBM's
  **additive + pairwise + bagging** is genuinely the stronger small-sample learner here.
- **The floor is real.** Every post-floor lever died for the *right* reason: the **log-signature**
  antisymmetric path basis is null (the close edge has **no chronological-order content** — a pure
  state/level regime), the turnover-**OFI** proxy is null, and five **regime-persistence** families are
  null OOS (best −0.00002; the cleanest in-sample axis, the fast/slow vol cascade, is 87% spanned).

⇒ **`0.12033` is the ceiling** of the price-only cascade. The two open levers are **(1) new data**
(auction imbalance / GEX) and **(2) DL on the linbest residual** targeting state-space (not path)
nonlinearity — exactly what ch. 04 tests by swapping the regime learner for a learned mixture-of-experts.